# M11.4 · Modelden bağımsız değerlendirme — Monte Carlo ve TD

Geçiş ve ödül modeli bilinmiyorsa değer, oynayarak kestirilir. Monte Carlo: bölüm sonuna kadar oyna, getiriyi ortala (yansız, yüksek varyans); zamansal fark (TD): bir adım oyna, $r+\gamma V(s')$'ye doğru güncelle (yanlı, düşük varyans). Öğrenme oranı koşulları, toplu (batch) MC ve TD'nin farklı cevaplara yakınsaması, n adımlı getiriler ve uygunluk izleriyle TD(λ). Saf Python'da 5 durumlu rastgele yürüyüşte MC ve TD; NumPy'da toplu güncelleme ve Sutton'ın A–B örneği; PyTorch'ta ileri bakışlı λ-getirisi ile geriye bakışlı izlerin çevrimdışı eşdeğerliği; en sonda 1000 bağımsız deneyi aynı anda yürüten vektörel TD.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-4-mc-td.html](https://m-gonul.github.io/pytorch-lab/m11-4-mc-td.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_4_mc_td.py
CHECKS = {
 "b": "import random\nimport numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def ep():\n        e = need(ns, \"walk_episode\")(random.Random(1))\n        ensure(e[0][0] == 3 and e[-1][2] in (0, 6) and all(abs(a - c) == 1 for a, _, c in e), \"geçişler\")\n        ensure(all(r == (1.0 if s2 == 6 else 0.0) for _, r, s2 in e), \"ödüller\")\n    R_.test(\"walk_episode\", ep)\n\n    def cmp():\n        runs = 100\n        curves = {}\n        for name, fn, a in [(\"MC α = 0.02\", \"mc\", 0.02), (\"MC α = 0.04\", \"mc\", 0.04), (\"TD α = 0.05\", \"td0\", 0.05), (\"TD α = 0.1\", \"td0\", 0.1)]:\n            acc = np.zeros(100)\n            for k in range(runs):\n                rng = random.Random(k)\n                eps = [ns[\"walk_episode\"](rng) for _ in range(100)]\n                _, h = need(ns, fn)(eps, a)\n                acc += np.array(h)\n            curves[name] = acc / runs\n        ensure(curves[\"TD α = 0.05\"][-1] < curves[\"MC α = 0.02\"][-1], \"100 bölümde TD MC'den düşük hata vermeli\")\n        R_.table(\"RMS hata, 100 deney ortalaması (Sutton Örnek 6.2)\", [\"yöntem\", \"10. bölüm\", \"50. bölüm\", \"100. bölüm\"],\n                 [[k, f\"{v[9]:.3f}\", f\"{v[49]:.3f}\", f\"{v[99]:.3f}\"] for k, v in curves.items()],\n                 note=\"TD bu Markov görevde MC'den hızlı öğrenir; büyük α'larda iki yöntemde de sabit bir gürültü tabanı kalır.\")\n    R_.test(\"MC ve TD öğrenme eğrileri\", cmp)\n\n    def one():\n        eps = [[(3, 0.0, 4), (4, 1.0, 6)]]\n        V, _ = ns[\"td0\"](eps, 0.5)\n        close(np.array(V[3:5]), np.array([0.5 + 0.5 * (0 + 0.5 - 0.5), 0.5 + 0.5 * (1 - 0.5)]), what=\"TD adımları\")\n        V, _ = ns[\"mc\"](eps, 0.5)\n        close(np.array(V[3:5]), np.array([0.75, 0.75]), what=\"MC adımları\")\n    R_.test(\"tek bölümde güncellemeler\", one)\n",
 "c": "import random\nimport numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef walk(rng):\n    s, out = 3, []\n    while True:\n        s2 = s + (1 if rng.random() < 0.5 else -1)\n        out.append((s - 1, 1.0 if s2 == 6 else 0.0))\n        if s2 in (0, 6):\n            return out\n        s = s2\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız tekrar döngüsü\", lambda: (max_loops(src, 0, func=\"batch_mc\"), max_loops(src, 1, func=\"batch_td\")))\n    AB = [[(0, 0.0), (1, 0.0)]] + [[(1, 1.0)]] * 6 + [[(1, 0.0)]]\n\n    def ab():\n        close(need(ns, \"batch_mc\")(AB, 2), np.array([0.0, 0.75]), what=\"toplu MC\")\n        close(need(ns, \"batch_td\")(AB, 2), np.array([0.75, 0.75]), atol=1e-6, what=\"toplu TD\")\n        close(need(ns, \"certainty_equivalence\")(AB, 2), np.array([0.75, 0.75]), atol=1e-12, what=\"kesinlik eşdeğerliği\")\n    R_.test(\"Sutton Örnek 6.4\", ab)\n\n    def rw():\n        true = np.arange(1, 6) / 6\n        rows = []\n        for k in (5, 20, 80):\n            mc_e = td_e = 0\n            for seed in range(30):\n                rng = random.Random(seed)\n                eps = [walk(rng) for _ in range(k)]\n                vt = ns[\"batch_td\"](eps, 5, 0.002, 1e-9)\n                close(vt, ns[\"certainty_equivalence\"](eps, 5), atol=1e-5, what=\"toplu TD = kesinlik eşdeğerliği\")\n                mc_e += np.sqrt(((ns[\"batch_mc\"](eps, 5) - true) ** 2).mean()); td_e += np.sqrt(((vt - true) ** 2).mean())\n            rows.append([k, f\"{mc_e / 30:.3f}\", f\"{td_e / 30:.3f}\"])\n        ensure(float(rows[0][2]) < float(rows[0][1]), \"az veride toplu TD daha doğru olmalı\")\n        R_.table(\"Toplu güncelleme, rastgele yürüyüş (30 tekrar)\", [\"bölüm\", \"toplu MC RMS\", \"toplu TD RMS\"], rows)\n    R_.test(\"rastgele yürüyüşte toplu MC vs TD\", rw)\n",
 "d": "import random\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef episode(rng, N=19):\n    s = N // 2 + 1\n    st, rw = [s], []\n    while True:\n        s2 = s + (1 if rng.random() < 0.5 else -1)\n        rw.append(1.0 if s2 == N + 1 else (-1.0 if s2 == 0 else 0.0))\n        st.append(s2)\n        if s2 in (0, N + 1):\n            return st, rw\n        s = s2\n\n\ndef check(ns, R_):\n    rng = random.Random(0)\n    V0 = torch.randn(21, dtype=torch.float64, generator=torch.Generator().manual_seed(0)) * 0.3\n    V0[0] = V0[20] = 0\n\n    def lr():\n        st, rw = [3, 4, 5, 6], [0.0, 0.0, 1.0]\n        V = torch.tensor([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0], dtype=torch.float64)\n        G = need(ns, \"lambda_returns\")(st, rw, V, 1.0, 0.0)\n        close(G.numpy(), torch.tensor([0.4, 0.5, 1.0]).numpy(), what=\"λ = 0: TD(0) hedefleri\")\n        G1 = ns[\"lambda_returns\"](st, rw, V, 1.0, 1.0)\n        close(G1.numpy(), torch.tensor([1.0, 1.0, 1.0]).numpy(), what=\"λ = 1: MC getirileri\")\n    R_.test(\"lambda_returns\", lr)\n\n    def eq():\n        for lam in (0.0, 0.4, 0.9, 1.0):\n            for k in range(5):\n                st, rw = episode(rng)\n                a = need(ns, \"offline_forward\")(st, rw, V0, 0.1, 0.95, lam)\n                b = need(ns, \"offline_traces\")(st, rw, V0, 0.1, 0.95, lam)\n                close(a.numpy(), b.numpy(), atol=1e-12, what=f\"λ = {lam}: ileri = geri (çevrimdışı)\")\n    R_.test(\"çevrimdışı ileri bakış = geriye bakış izleri\", eq)\n\n    def on():\n        st, rw = episode(random.Random(5))\n        a = ns[\"offline_traces\"](st, rw, V0, 0.1, 1.0, 0.8)\n        b = need(ns, \"online_traces\")(st, rw, V0, 0.1, 1.0, 0.8)\n        d = float((a - b).abs().max())\n        ensure(d > 0, \"çevrimiçi sürüm farklı olmalı\")\n        rows = []\n        true = torch.linspace(-1, 1, 21, dtype=torch.float64)[1:20] * 1.0\n        true = torch.arange(1, 20, dtype=torch.float64) / 10 - 1\n        for lam in (0.0, 0.4, 0.8, 0.95, 1.0):\n            err = 0\n            for seed in range(20):\n                r2 = random.Random(seed)\n                V = torch.zeros(21, dtype=torch.float64)\n                for _ in range(10):\n                    st, rw = episode(r2)\n                    V = ns[\"online_traces\"](st, rw, V, 0.2, 1.0, lam)\n                err += float(((V[1:20] - true) ** 2).mean().sqrt())\n            rows.append([lam, f\"{err / 20:.3f}\"])\n        R_.table(\"19 durumlu yürüyüş, 10 bölüm, α = 0.2, çevrimiçi izler\", [\"λ\", \"RMS\"], rows,\n                 note=f\"Çevrimiçi ve çevrimdışı sonuç aynı bölümde {d:.2e} farklı. Ara λ en iyidir; λ = 1'de büyük α ile kararsızlaşır (Sutton Şekil 12.3).\")\n    R_.test(\"online_traces ve λ eğrisi\", on)\n",
 "e": "import random\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef loop_td(n_runs, n_steps, alpha, seed=0):\n    out = []\n    for k in range(n_runs):\n        rng = random.Random(seed * 1000 + k)\n        V = [0.0] + [0.5] * 5 + [0.0]\n        s = 3\n        for _ in range(n_steps):\n            s2 = s + (1 if rng.random() < 0.5 else -1)\n            V[s] += alpha * ((1.0 if s2 == 6 else 0.0) + V[s2] - V[s])\n            s = 3 if s2 in (0, 6) else s2\n        out.append(V)\n    return out\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"batched_td\"))\n\n    def t():\n        V, rms = need(ns, \"batched_td\")(1000, 3000, 0.02, 5, 1)\n        ensure(tuple(V.shape) == (1000, 7) and rms.shape[0] == 3000, \"şekiller\")\n        close(V[:, 1:6].mean(0).numpy(), (torch.arange(1, 6, dtype=torch.float64) / 6).numpy(), atol=0.02, what=\"ortalama V ≈ gerçek değer\")\n        ensure(float(rms[-1]) < 0.06 and float(rms[-1]) < float(rms[100]), \"hata azalmalı\")\n        ref = loop_td(200, 3000, 0.02)\n        refm = torch.tensor(ref).mean(0)\n        close(V[:200].mean(0).numpy(), refm.numpy(), atol=0.03, what=\"döngülü uygulamayla istatistiksel uyum\")\n        t1 = bench(lambda: loop_td(200, 3000, 0.02), repeat=1, number=1) * 5\n        t2 = bench(lambda: ns[\"batched_td\"](1000, 3000, 0.02), repeat=1, number=1)\n        R_.table(\"1000 deney × 3000 adım\", [\"yöntem\", \"süre\"], [[\"deney döngüsü (saf Python, tahmini)\", fmt_time(t1)], [\"tensör (deneyler paralel)\", fmt_time(t2)]])\n    R_.test(\"batched_td\", t)\n",
 "x1": "import random\nimport numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = random.Random(3)\n        for _ in range(200):\n            st, r = need(ns, \"play\")(rng)\n            ensure(r in (-1, 0, 1) and all(12 <= s[1] <= 21 and 1 <= s[2] <= 10 for s in st), \"durum aralıkları\")\n        V = need(ns, \"mc_blackjack\")(30000, 1)\n        ensure(V.shape == (2, 10, 10) and np.abs(V).max() <= 1, \"şekil ve aralık\")\n        ensure(V[0, 8:, :].mean() > 0.3 and V[0, :8, :].mean() < -0.2, \"20–21'de durmak kazandırır, 12–19'da çekmek riskli\")\n        R_.table(\"Kullanılabilir as yok, 30 000 el (Sutton Şekil 5.1)\", [\"oyuncu toplamı\", \"krupiye A\", \"krupiye 6\", \"krupiye 10\"],\n                 [[t + 12, f\"{V[0, t, 0]:+.2f}\", f\"{V[0, t, 5]:+.2f}\", f\"{V[0, t, 9]:+.2f}\"] for t in (0, 4, 8, 9)])\n    R_.test(\"blackjack MC\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        res = {}\n        for name, f in [(\"1/n^0.6\", lambda n: n ** -0.6), (\"1/n\", lambda n: 1 / n), (\"sabit 0.1\", lambda n: 0.1), (\"1/n²\", lambda n: 1 / n ** 2)]:\n            res[name] = np.mean([need(ns, \"run\")(f, 2000, -3.0, s) for s in range(10)], 0)\n        ensure(res[\"1/n^0.6\"][-1] < 0.05, \"1/n^0.6 yakınsamalı\")\n        ensure(res[\"1/n\"][-1] < res[\"1/n\"][99], \"1/n yavaş da olsa iyileşmeli\")\n        ensure(res[\"1/n²\"][-1] > 1.0, \"1/n² kötü başlangıçtan kurtulamamalı\")\n        ensure(res[\"sabit 0.1\"][-1] > res[\"1/n^0.6\"][-1], \"sabit α gürültü tabanında kalmalı\")\n        R_.table(\"V0 = −3 (gerçek değerler 0.17–0.83), 10 deney\", [\"α_n\", \"100. bölüm\", \"2000. bölüm\"], [[k, f\"{v[99]:.3f}\", f\"{v[-1]:.3f}\"] for k, v in res.items()],\n                 note=\"Σα = ∞ başlangıç hatasını unutmayı, Σα² < ∞ gürültünün sönmesini sağlar. Koşullar yakınsamayı garanti eder, hızı değil: 1/n TD'de logaritmik hızla unutur.\")\n    R_.test(\"Robbins–Monro koşulları\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        best = {}\n        for n in (1, 2, 4, 8, 16):\n            row = [n]\n            for a in (0.1, 0.2, 0.4, 0.8):\n                e = np.mean([need(ns, \"nstep_td\")(need(ns, \"episodes19\")(10, s), n, a) for s in range(20)])\n                row.append(f\"{e:.3f}\"); best[(n, a)] = e\n            rows.append(row)\n        n_best = min(best, key=best.get)\n        ensure(1 < n_best[0] < 16, f\"en iyi n arada olmalı, bulunan {n_best}\")\n        R_.table(\"19 durumlu yürüyüş, 10 bölüm, 20 tekrar ortalama RMS\", [\"n\", \"α = 0.1\", \"α = 0.2\", \"α = 0.4\", \"α = 0.8\"], rows,\n                 note=f\"En iyi (n, α) = {n_best}: ne tek adım ne de MC — ara n en iyidir (Sutton Şekil 7.2).\")\n    R_.test(\"n ve α taraması\", t)\n",
 "x4": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t_sel():\n        rng = np.random.default_rng(0)\n        sel = need(ns, \"select\")\n        q = np.array([0.0, 1.0, 1.0])\n        picks = [sel(q, \"greedy\", 0, rng) for _ in range(400)]\n        ensure(set(picks) == {1, 2}, \"greedy: eşitlikleri rastgele boz (hep ilkini seçme)\")\n        picks = np.array([sel(np.array([0.0, 1.0]), \"eps\", 0, rng, eps=0.5) for _ in range(4000)])\n        ensure(abs((picks == 0).mean() - 0.25) < 0.03, \"ε = 0.5: kötü eylem ≈ ε/2 olasılıkla\")\n        picks = np.array([sel(np.array([0.0, 0.1]), \"boltzmann\", 0, rng, tau=0.1) for _ in range(4000)])\n        ensure(abs((picks == 1).mean() - 1 / (1 + np.exp(-1))) < 0.03, \"Boltzmann: p ∝ exp(q/τ)\")\n        early = np.array([sel(np.array([0.0, 1.0]), \"eps-decay\", 0, rng) for _ in range(2000)])\n        late = np.array([sel(np.array([0.0, 1.0]), \"eps-decay\", 400, rng) for _ in range(2000)])\n        ensure((early == 0).mean() > 0.4 and (late == 0).mean() < 0.06, \"azalan ε: başta ≈ 1, sonra 0.05\")\n    R_.test(\"eylem seçimi\", t_sel)\n\n    def t_chain():\n        ql = need(ns, \"q_learning\")\n        rows, res = [], {}\n        for name, m, q0 in ((\"ε-greedy, ε = 0.1\", \"eps\", 0.0), (\"azalan ε (1 → 0.05)\", \"eps-decay\", 0.0), (\"Boltzmann, τ = 0.1\", \"boltzmann\", 0.0), (\"iyimser başlangıç Q₀ = 20, greedy\", \"greedy\", 20.0)):\n            r = [name]\n            for N in (6, 10):\n                out = [ql(N, 250, m, q0, s) for s in range(8)]\n                found = np.mean([f is not None for _, f in out])\n                res[(m, N)] = found\n                r += [f\"{found:.2f}\", f\"{np.mean([g[-50:].mean() for g, _ in out]):.2f}\"]\n            rows.append(r)\n        ensure(res[(\"greedy\", 10)] == 1.0 and res[(\"eps\", 10)] < 0.5, f\"uzun zincirde iyimser başlangıç bulur, sabit ε çoğunlukla bulamaz {res}\")\n        R_.table(\"Zincir ortamında keşif (8 tohum, 250 bölüm): uzak ödülü bulma oranı ve son 50 bölümün ortalama getirisi\", [\"yöntem\", \"N = 6: buldu\", \"N = 6: getiri\", \"N = 10: buldu\", \"N = 10: getiri\"], rows,\n                 note=\"Soldaki küçük ödül (0.05) hemen bulunur ve açgözlü politikayı oraya kilitler. Sağ uca ulaşmak için art arda N − 2 kez sağa gitmek gerekir: rastgele (yönsüz) keşifle olasılık ≈ 2^−(N−2), zincir uzadıkça üstel olarak küçülür. İyimser başlangıç denenmemiş her eylemi 'cazip' kılar: ajan sistematik olarak dener, değer gerçekleri gördükçe iner (belirsizlik karşısında iyimserlik — UCB'nin de ilkesi, M11.7). Sabit ε'lu ajan uzak ödülü bulsa bile ε oranında rastgele davrandığı için getirisi düşük kalır.\")\n    R_.test(\"keşif yöntemleri\", t_chain)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_4_mc_td as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Kurulum:** politika $\pi$ sabit, $P$ ve $R$ **bilinmiyor**. Yalnız oynayarak deneyim toplayabiliriz. (Bisiklete binmeyi fiziği bilmeden öğrenmek.) Ajan bileşenleri: değer tabanlı, politika tabanlı, aktör–eleştirmen; modelli ya da modelsiz.

**Monte Carlo değerlendirme:** bölümü sonuna kadar oyna, her ziyaret edilen durum için getiriyi hesapla $G_t=r_t+\gamma r_{t+1}+\cdots$, ortala. Artımlı: $V(s)\leftarrow V(s)+\alpha_n(s)\,[G-V(s)]$, $\alpha_n=1/n(s)$ ile tam ortalama. İlk ziyaret / her ziyaret. Yansız, ama getiri birçok rastgele adımın toplamı olduğu için **varyansı yüksek**; çok bölüm ister ve yalnız bölümlü görevlerde çalışır.

**TD(0):** tek adım oyna, kalan getiriyi kendi kestirimimizle tamamla (önyükleme):

$$V(s)\leftarrow V(s)+\alpha\,\big[\underbrace{r+\gamma V(s')-V(s)}_{\delta}\big].$$

Yanlı (V yanlışken hedef de yanlış), **düşük varyans**, her adımda öğrenir. Yakınsama (tablo) Robbins–Monro koşullarıyla: $\sum_n\alpha_n=\infty$, $\sum_n\alpha_n^2<\infty$ (ör. $\alpha_n=1/n$).

**Toplu güncelleme:** sınırlı veriyi tekrar tekrar sunarsak MC ve TD **farklı** cevaplara yakınsar: MC eğitim verisindeki getirilerin ortalamasını, TD ise verinin en çok olabilirlikli Markov modelinin değerini ("kesinlik eşdeğerliği"). Markov ortamlarda TD genelde daha iyi genelleştirir.

**Arası:** n adımlı getiri $G^{(n)}_t=r_t+\cdots+\gamma^{n-1}r_{t+n-1}+\gamma^nV(s_{t+n})$; λ-getirisi $G^\lambda_t=(1-\lambda)\sum_n\lambda^{n-1}G^{(n)}_t$. Geriye bakış: **uygunluk izi** $e(s)\leftarrow\gamma\lambda e(s)+\mathbb 1[s=s_t]$, her adımda $V\leftarrow V+\alpha\delta_te$. λ = 0 → TD(0), λ = 1 → MC.

:::tanım Bir RL ajanının bileşenleri ve ajan türleri
Bir ajan şunlardan bir ya da birkaçını içerir: **politika** $\pi(a|s)$ (ne yapacağı), **değer fonksiyonu** $V(s)$ ya da $Q(s,a)$ (ne kadar iyi olduğu), **model** $\hat P(s'|s,a)$, $\hat r(s,a)$ (ortamın nasıl davrandığı). Buna göre: **değer tabanlı** (Q öğrenir, politika örtük: $\arg\max_aQ$; SARSA, Q-öğrenme, DQN), **politika tabanlı** (politikayı doğrudan parametreleştirir; REINFORCE), **aktör–eleştirmen** (ikisi birden; M11.9); ayrıca **modelsiz** (deneyimden doğrudan) ve **modele dayalı** (modeli öğrenip planlar; Dyna, MCTS — M11.10). Bu modülün MC ve TD yöntemleri modelsiz değer tahminidir; kontrol için **keşif** gerekir: hiç denenmemiş bir eylemin değeri öğrenilemez. Ek alıştırma E4 keşif yöntemlerini (ε-greedy, azalan ε, Boltzmann/softmax, iyimser başlangıç) aynı problemde karşılaştırır.
:::

## 1 · Elle

<!-- çözüm -->

TD: $\delta=0+0.7-0.5=0.2$, $V\leftarrow0.5+0.1\cdot0.2=0.52$.

A–B: B 8 kez görüldü, 6'sında 1 → 0.75. A bir kez görüldü ve getirisi 0 → MC: 0. TD: A'dan hep B'ye geçilmiş ve ödül 0 → $V(A)=0+V(B)=0.75$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Güncellemeler**

- `inc`: Artımlı ortalama: $V=0.5$ (3 örnekten), yeni $G=1$, $\alpha=1/4$
- `mc`: Sabit α MC: $V=0.5$, $G=1$, $\alpha=0.1$
- `td`: TD(0): $V(s)=0.5$, $r=0$, $V(s')=0.7$, $\gamma=1$, $\alpha=0.1$
- `dlt`: Aynı adımın TD hatası δ

**Sutton Örnek 6.4: bölümler A,0,B,0 · 6 × (B,1) · 1 × (B,0)**

- `vb`: $V(B)$ (ikisi de)
- `amc`: Toplu MC'nin $V(A)$'sı
- `atd`: Toplu TD'nin $V(A)$'sı

**5 durumlu rastgele yürüyüş (A…E, sağ uç +1)**

- `rc`: C'nin gerçek değeri

In [ ]:
cevaplar = dict(
    inc=None,
    mc=None,
    td=None,
    dlt=None,
    vb=None,
    amc=None,
    atd=None,
    rc=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"inc":[0.625,0],"mc":[0.55,0],"td":[0.52,0],"dlt":[0.2,0],"vb":[0.75,0],"amc":[0,0],"atd":[0.75,0],"rc":[0.5,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. **Rastgele yürüyüş:** durumlar 1..5 (A..E), 0 ve 6 terminal; C'den (3) başla, her adım ±1 eşit olasılıkla; 6'ya ulaşınca ödül 1, aksi halde 0; γ = 1. Gerçek değerler $i/6$.

- `walk_episode(rng)` — `random.Random` ile bir bölüm: `[(s, r, s2), …]` geçiş listesi.
- `mc(episodes, alpha, V0=0.5)` — her ziyaret, sabit α MC; her bölümden sonra RMS hatayı kaydet. `(V, rms_list)` — `V` 7 elemanlı (terminaller 0).
- `td0(episodes, alpha, V0=0.5)` — TD(0), her geçişte güncelle; aynı çıktı.
- `rms(V)` — 1..5 üzerinde gerçek değerlere RMS.

In [ ]:
%%writefile stage_b.py
import math
import random


def walk_episode(rng):
    ...


def mc(episodes, alpha, V0=0.5):
    ...


def td0(episodes, alpha, V0=0.5):
    ...


def rms(V):
    ...


rng = random.Random(0)
eps = [walk_episode(rng) for _ in range(100)]
print(rms(td0(eps, 0.1)[0]), rms(mc(eps, 0.02)[0]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Toplu (batch) güncelleme, NumPy, yalnız tekrar döngüsü serbest.

- `batch_mc(episodes, n_states)` — her durum için gözlenen tüm getirilerin ortalaması (ilk ziyaret değil, **her ziyaret**); `episodes` her biri `[(s, r), …]` ve terminale geçiş örtük. Getiri: ters `cumsum` (γ = 1).
- `batch_td(episodes, n_states, alpha=0.01, tol=1e-10)` — tüm geçişleri `(s, r, s2)` dizilerine diz (`s2 = -1` terminal), her taramada artımları **topla** ve birlikte uygula: $\Delta V=\alpha\sum\delta$ (`np.add.at`); değişim < tol olana dek.
- `certainty_equivalence(episodes, n_states)` — sayımlardan $\hat P$, $\hat R$ (terminale geçiş olasılığı dahil), $V=(I-\hat P)^{-1}\hat r$: toplu TD ile aynı olmalı.

In [ ]:
%%writefile stage_c.py
import numpy as np


def batch_mc(episodes, n_states):
    ...


def batch_td(episodes, n_states, alpha=0.01, tol=1e-10):
    ...


def certainty_equivalence(episodes, n_states):
    ...


AB = [[(0, 0.0), (1, 0.0)]] + [[(1, 1.0)]] * 6 + [[(1, 0.0)]]
print(batch_mc(AB, 2), batch_td(AB, 2))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

TD(λ): ileri bakış ve geriye bakış (PyTorch, tek bölüm, `float64`).

- `lambda_returns(states, rewards, V, gamma, lam)` — bölüm `states[0..T]` (son durum terminal, değeri 0), `rewards[0..T-1]`; her $t$ için $G^\lambda_t$ (özyineleme: $G^\lambda_t=r_t+\gamma[(1-\lambda)V(s_{t+1})+\lambda G^\lambda_{t+1}]$, $G^\lambda_{T}=0$).
- `offline_forward(states, rewards, V, alpha, gamma, lam)` — $\Delta V(s_t)\mathrel{+}=\alpha(G^\lambda_t-V(s_t))$, hepsi bölüm sonunda uygulanır.
- `offline_traces(states, rewards, V, alpha, gamma, lam)` — birikimli izler $e\leftarrow\gamma\lambda e+\mathbf 1_{s_t}$, $\Delta V\mathrel{+}=\alpha\delta_te$ (δ'lar **eski** V ile), bölüm sonunda uygulanır.
- `online_traces(...)` — aynı ama V her adımda güncellenir.

Çevrimdışı ileri ve geriye bakış **tam olarak** eşittir (Sutton & Barto 12.1).

In [ ]:
%%writefile stage_d.py
import torch


def lambda_returns(states, rewards, V, gamma, lam):
    ...


def offline_forward(states, rewards, V, alpha, gamma, lam):
    ...


def offline_traces(states, rewards, V, alpha, gamma, lam):
    ...


def online_traces(states, rewards, V, alpha, gamma, lam):
    ...


V = torch.zeros(7, dtype=torch.float64)
print(offline_traces([3, 4, 5, 6], [0.0, 0.0, 1.0], V, 0.1, 1.0, 0.8))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

1000 bağımsız deney, aynı anda (PyTorch).

- `batched_td(n_runs, n_steps, alpha, n=5, seed=0)` — `n` durumlu rastgele yürüyüş (1..n, uçlar terminal, sağ uç ödül 1); her deney kendi konumunda; her adımda tüm deneyler bir adım atar, TD(0) güncellemesi `V[run, s]` üzerinde (`scatter_add_` ya da gelişmiş indeksleme — bir deneyde bir adımda tek durum güncellendiği için çakışma yok); terminale ulaşan deney merkeze sıfırlanır. `(V (n_runs, n+2), rms (n_steps,))` — her adımdan sonra deneyler arası ortalama RMS.

Adım döngüsü serbest; deney döngüsü yok.

In [ ]:
%%writefile stage_e.py
import torch


def batched_td(n_runs, n_steps, alpha, n=5, seed=0):
    ...


V, rms = batched_td(100, 500, 0.1)
print(V.shape, rms[-1])

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Blackjack: Monte Carlo ile politika değerlendirme

**Blackjack** (Sutton Örnek 5.1): sonsuz deste (kartlar 1–10, 10 olasılığı 4/13), oyuncu 20 ya da 21'de durur, aksi halde kart çeker; krupiye 17 ve üstünde durur. Durum: (kullanılabilir as, oyuncu toplamı 12–21, krupiyenin açık kartı 1–10). Kazanç +1, beraberlik 0, kayıp −1.

- `play(rng)` → `(states, reward)`: bir el; toplam 12'nin altındaysa oyuncu otomatik çeker (durum kaydedilmez).
- `mc_blackjack(n, seed=0)` → `V (2, 10, 10)` (as, toplam−12, krupiye−1), her ziyaret ortalaması.

In [ ]:
%%writefile extra_x1.py
import random
import numpy as np


def draw(rng):
    return min(rng.randint(1, 13), 10)


def play(rng):
    ...


def mc_blackjack(n, seed=0):
    ...


print(mc_blackjack(1000)[0, 9, 0])

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Öğrenme oranı koşulları deneyi

- `run(alpha_fn, n_episodes, V0, seed=0)` — 5 durumlu yürüyüşte TD(0), durum başına sayaçla `alpha = alpha_fn(n(s))`; RMS listesi.

Kontrol: $1/n^{0.6}$ (koşullar sağlanır, hızlı yakınsar), $1/n$ (koşullar sağlanır ama TD'de çok yavaş: ilk, yanlış önyükleme hedefleri sonsuza dek eşit ağırlıkla ortalamada kalır), sabit 0.1 (gürültü tabanı kalır), $1/n^2$ ($\sum\alpha<\infty$: kötü başlangıçtan kurtulamaz).

In [ ]:
%%writefile extra_x2.py
import random
import math


def run(alpha_fn, n_episodes, V0, seed=0):
    ...


print(run(lambda n: 1 / n, 50, 0.0)[-1])

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · n adımlı TD: 19 durumlu yürüyüşte n ve α

- `nstep_td(episodes, n, alpha, N=19, gamma=1.0)` — 19 durumlu yürüyüş (sol uç −1, sağ uç +1, gerçek değerler $-1..1$ doğrusal), n adımlı TD (Sutton Alg. 7.1), V başlangıç 0; son RMS.
- `episodes19(k, seed=0)` — k bölüm (merkezden başlar).

In [ ]:
%%writefile extra_x3.py
import random
import math


def episodes19(k, seed=0):
    ...


def nstep_td(episodes, n, alpha, N=19, gamma=1.0):
    ...


print(nstep_td(episodes19(10), 4, 0.4))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

### E4 · Keşif yöntemleri: ε-greedy, azalan ε, Boltzmann, iyimser başlangıç

Kontrolde değer tahmini yetmez: hiç denenmemiş bir eylemin değeri öğrenilemez. **Keşif yöntemleri** (NumPy):

- `select(q, method, t, rng, eps=0.1, tau=0.1)` — `q` bir durumun eylem değerleri; `"greedy"` (eşitlikleri **rastgele** boz), `"eps"` (ε olasılıkla düzgün rastgele), `"eps-decay"` ($\varepsilon_t=\max(0.05,\,1-t/200)$, t bölüm sayısı), `"boltzmann"` ($p(a)\propto e^{q_a/\tau}$; büyük değeri çıkararak hesapla).
- `q_learning(N, episodes, method, q0, seed, alpha=0.2, gamma=0.95)` → `(bölüm getirileri, uzak ödülün ilk bulunduğu bölüm ya da None)`: Q tablosu `q0` ile başlar; her bölüm durum 1'den başlar, N + 6 adım sürer; güncelleme $Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$ (M11.5'te ayrıntılı). Verilen: `chain_step(s, a, N)`.

**İyimser başlangıç:** Q'yu olası en büyük getiriden büyük başlat ($Q_0=20>1/(1-\gamma)$) ve tamamen açgözlü davran — her denenmemiş eylem gerçek değerinden iyi göründüğü için bir kez denenir.

In [ ]:
%%writefile extra_x4.py
import numpy as np


def chain_step(s, a, N):
    """Zincir: a = 0 sola, 1 sağa. Solda (s = 0) sola gitmek 0.05 ödül, sağ uçta (s = N−1) sağa gitmek 1 ödül."""
    if a == 0:
        return max(s - 1, 0), (0.05 if s == 0 else 0.0)
    return min(s + 1, N - 1), (1.0 if s == N - 1 else 0.0)


def select(q, method, t, rng, eps=0.1, tau=0.1):
    ...


def q_learning(N, episodes, method, q0, seed, alpha=0.2, gamma=0.95):
    ...

In [ ]:
# Kontrol (E4)
labcheck.run_file("extra_x4.py", CHECKS["x4"])

## Makale ödevi: Sutton (1988): TD yöntemleriyle tahmin

[Sutton — Learning to Predict by the Methods of Temporal Differences (Machine Learning 1988)](https://doi.org/10.1007/BF00115009)

Makalenin rastgele yürüyüş deneylerini (Şekil 3, 4, 5) yeniden üret: 100 eğitim kümesi × 10 dizi, (1) tekrarlı sunum (batch, yakınsayana dek) altında λ'ya karşı RMS hata, (2) tek sunum altında farklı α ve λ için hata. Doğrusal fonksiyon yaklaştırıcı (birim vektör öznitelikleri) ile Widrow–Hoff kuralının TD(1) olduğunu göster.


**Adımlar**

1. Bu modüldeki d aşamasının iz uygulamasını kullan; ağırlıkları dizi sonunda güncelle.
2. Tekrarlı sunumda ağırlık değişimi < 1e-3 olana dek tekrar.
3. En iyi α ile λ ∈ {0, 0.1, …, 1} eğrisi.

**Kabul ölçütleri**

- Şekil 3: hata λ arttıkça artar (λ = 0 en iyi, ≈ 0.19), λ = 1 ≈ 0.25.
- Şekil 4/5'in nitel biçimi: ara λ (≈ 0.3) tek sunumda en iyi.

## Kaynaklar

- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 5–7 ve 12](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Sutton — Learning to Predict by the Methods of Temporal Differences (Machine Learning 1988)](https://doi.org/10.1007/BF00115009) — ©
- [Robbins & Monro — A Stochastic Approximation Method (Ann. Math. Stat. 1951)](https://doi.org/10.1214/aoms/1177729586) — ©
- [Poupart — CS885 Reinforcement Learning, Ders 4](https://cs.uwaterloo.ca/~ppoupart/teaching/cs885-spring20/) — doğrulanmadı